# example / onnx / linear

**Intent:** For an input number (`example_onnx_in`), output the input scaled by a fitted gain plus a fitted offset (`example_onnx_out`), built as an ONNX graph, to show an `onnx` project end to end.

**Notes:**
- Purpose: Show an `onnx` model end to end: fit, build the graph, accept, replay the golden and register
- Data splits: `training` ten labelled points on a line with fixed noise, to fit; `validation` five more; `backtest` ten further out, held back for acceptance; `golden` the exact samples that become `golden.json`

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, version and dataset names as constants (`model.md` 6.6).

In [ ]:
import json
from os.path import join

import numpy as np

from workbench.config import NOTEBOOK_DIR
from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import load_signature, provenance, record_project, scratch_project_dir, signature_names

env = init_env()

WORKSPACE = "example/onnx/linear"
MODEL_NAME = "example_onnx_linear"
TARGETS = [{"kind": "served"}, {"kind": "embedded", "language": "python"}]
signature = load_signature(WORKSPACE)
INPUTS, OUTPUTS = signature_names(signature)
TRAINING = "onnx_linear_training_2026-11-03_2026-11-03"
VALIDATION = "onnx_linear_validation_2026-11-03_2026-11-03"
BACKTEST = "onnx_linear_backtest_2026-11-03_2026-11-03"
GOLDEN = "onnx_linear_golden"
INPUT = INPUTS[0]
OUTPUT = OUTPUTS[0]

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training`, `validation` and `backtest` files only when they are absent (`model.md` 6.5). The source is synthetic, so no query runs.

In [ ]:
NOISE = [0.04, -0.03, 0.02, -0.05, 0.01, 0.03, -0.02, 0.05, -0.04, 0.0]


def labelled(first, count):
    xs = [float(x) for x in range(first, first + count)]
    return {INPUT: xs, OUTPUT: [round(2.5 * x + 1.0 + NOISE[int(x) % 10], 4) for x in xs]}


training_pl, training_path = ensure_data(WORKSPACE, "training", TRAINING, lambda: labelled(0, 10))
validation_pl, validation_path = ensure_data(WORKSPACE, "validation", VALIDATION, lambda: labelled(10, 5))
backtest_pl, backtest_path = ensure_data(WORKSPACE, "backtest", BACKTEST, lambda: labelled(15, 10))
print("training", training_pl.height, "validation", validation_pl.height, "backtest", backtest_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** fit a straight line on `training` only (R6). Its gain and offset are rounded to one place, which the graph builds in.

In [ ]:
gain, offset = np.polyfit(training_pl[INPUT].to_numpy(), training_pl[OUTPUT].to_numpy(), 1)
GAIN, OFFSET = round(float(gain), 1), round(float(offset), 1)
print("fitted gain", GAIN, "offset", OFFSET)
assert (GAIN, OFFSET) == (2.5, 1.0), "the fit did not recover the line that made the data"

## Evaluate

- **Stage:** 4, evaluate.
- **Intent:** write the graph at a pinned opset from the fitted line, build the signature, run the graph through the library runtime and check the `validation` and `backtest` error against the limit the notebook owns (R4, R14).

In [ ]:
import onnx
from onnx import TensorProto, helper, numpy_helper

from workbench.model import evaluate_acceptance, measure
from workbench.replay import model_predict

ONNX_OPSET = 21
ONNX_IR_VERSION = 10

graph = helper.make_graph(
    [helper.make_node("Mul", [INPUT, "gain"], ["scaled"]), helper.make_node("Add", ["scaled", "offset"], [OUTPUT])],
    "linear",
    [helper.make_tensor_value_info(INPUT, TensorProto.DOUBLE, [1, 1])],
    [helper.make_tensor_value_info(OUTPUT, TensorProto.DOUBLE, [1, 1])],
    initializer=[numpy_helper.from_array(np.array(GAIN, dtype=np.float64), "gain"), numpy_helper.from_array(np.array(OFFSET, dtype=np.float64), "offset")],
)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", ONNX_OPSET)])
model.ir_version = ONNX_IR_VERSION
onnx.checker.check_model(model)

model_path = join(NOTEBOOK_DIR, WORKSPACE, "model.onnx")
onnx.save(model, model_path)

predict = model_predict(model_path, "onnx", signature)

In [ ]:
def acceptance_checks():
    checks = []
    for split, frame in (("validation", validation_pl), ("backtest", backtest_pl)):
        predicted = [predict({INPUT: [x]})[OUTPUT][0] for x in frame[INPUT].to_list()]
        checks.append({"split": split, "output": OUTPUT, "name": "rmse", "value": measure(frame[OUTPUT].to_list(), predicted)["rmse"], "limit": 0.5, "direction": "max"})
    return checks


evaluation = evaluate_acceptance(acceptance_checks, {"validation": [validation_path], "backtest": [backtest_path]})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "limit", entry["limit"], "passed", entry["passed"])

## Package

- **Stage:** 5, package.
- **Intent:** write the golden file, copy the graph and the evaluation into a project, verify it and replay the golden through the library runtime (R1, R3, R14).

In [ ]:
def golden_samples():
    xs = [-1.0, 0.0, 0.25, 1.0, 2.5, 10.0]
    return {INPUT: [*xs, None], OUTPUT: [*[GAIN * x + OFFSET for x in xs], None], "expect": ["ok"] * len(xs) + ["error"]}

In [ ]:
from workbench.model import package_project, register_preview
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, golden_samples)

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "onnx",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic line, training split",
        [training_path, golden_path],
        "least squares line, gain and offset built into the graph",
        implementation="onnx.helper graph, opset 21, ir version 10",
    ),
    model_path=model_path,
    evaluation=evaluation,
    tolerance={OUTPUT: {"absolute": 1e-9}},
    sequence="across zero, fractions and a larger value",
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped runtime")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview`. A person sets `release` after review and the parity gate (R5).

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS, metadata["snapshot_version"])
print("registered version", version)